# Compile metrics across all runs
 
This notebook loads MAGICC output for all model-scenario combinations and extracts
temperature metrics at key points:
- At net-zero CO2
- At 2100
 
 For the following temperature components:
 - ALL (total warming)
 - CO2 [NZCO2] (CO2 warming at net-zero)
 - CO2 [Negative] (warming from negative CO2 emissions)
 - Non-CO2 GHG
 - Residual

In [ ]:
import pandas as pd
import numpy as np
import pyam

from pathlib import Path
from tqdm import tqdm

from utils import (
    get_value_at_netzero,
    get_value_at_year
)
from metrics import (
    load_runs_manifest,
    load_scenario_data,
    extract_metrics,
    DEFAULT_VARIABLES,
    DEFAULT_VAR_SHORT_NAMES,
)

%load_ext autoreload
%autoreload 2

# Configuration
OUTPUT_DIR = Path("output/individual_runs")
NETZERO_META_FILE = Path("output/102_netzero_timings.csv")
MANIFEST_FILE = Path("output/individual_runs/manifests/2026-05-06_11-41_manifest.csv")

In [ ]:
# Load and pivot runs manifest
runs_pivoted = load_runs_manifest(MANIFEST_FILE)

print(f"Found {len(runs_pivoted)} unique model-scenario combinations")
runs_pivoted

In [ ]:
# Use default variable configurations from metrics.py
# These can be overridden if needed
VARIABLES = DEFAULT_VARIABLES
VAR_SHORT_NAMES = DEFAULT_VAR_SHORT_NAMES

In [6]:
# Load net-zero metadata
netzero_meta = pd.read_csv(NETZERO_META_FILE)

In [7]:
# Process all scenarios and compile metrics
all_metrics = []

for model, scenario in tqdm(runs_pivoted.index, desc="Processing scenarios"):
    try:
        # Load scenario data
        df = load_scenario_data(model, scenario, runs_pivoted, OUTPUT_DIR)
        
        # Load net-zero metadata
        df.load_meta(NETZERO_META_FILE)
        
        # Extract metrics
        metrics = extract_metrics(df, VARIABLES, VAR_SHORT_NAMES)
        
        if not metrics.empty:
            all_metrics.append(metrics)
            
    except Exception as e:
        print(f"Error processing {model} / {scenario}: {e}")
        continue

# Combine all metrics into single DataFrame
metrics_df = pd.concat(all_metrics, axis=0)
print(f"\nCompiled metrics for {len(metrics_df)} runs across {len(runs_pivoted)} scenarios")

Processing scenarios:   0%|          | 0/5 [00:00<?, ?it/s][INFO] 11:52:56 - pyam.core: Reading file output/individual_runs/IMAGE__3.0.1_EMF30-ClimatePolicy+SLCF_ALL.csv
[INFO] 11:52:56 - pyam.core: Reading file output/individual_runs/IMAGE__3.0.1_EMF30-ClimatePolicy+SLCF_CO2.csv
[INFO] 11:52:56 - pyam.core: Reading file output/individual_runs/IMAGE__3.0.1_EMF30-ClimatePolicy+SLCF_GHG.csv
[INFO] 11:52:56 - pyam.core: Reading file output/individual_runs/IMAGE__3.0.1_EMF30-ClimatePolicy+SLCF_NZCO2_CO2.csv
[WARNING] 11:52:58 - pyam.core: Ignoring meta indicators for the following scenarios:
         model       scenario
0  AIM/CGE 2.0        SSP1-19
1  AIM/CGE 2.0        SSP1-26
2  AIM/CGE 2.0        SSP1-34
3  AIM/CGE 2.0        SSP1-45
4  AIM/CGE 2.0  SSP1-Baseline
...
Processing scenarios:  20%|██        | 1/5 [00:02<00:09,  2.41s/it][INFO] 11:52:58 - pyam.core: Reading file output/individual_runs/MESSAGEix-GLOBIOM__1.1_ENGAGE-NPi2020-1000_ALL.csv
[INFO] 11:52:59 - pyam.core: Reading f


Compiled metrics for 3000 runs across 5 scenarios


In [8]:
# View the compiled metrics
metrics_df.round(2)

GSAT|ALL|NZCO2  GSAT|ALL|2100  \
model       scenario                    run_id                                  
IMAGE 3.0.1 EMF30-ClimatePolicy+SLCF    0                 2.00           2.03   
                                        1                 3.36           3.51   
                                        2                 2.14           2.17   
                                        3                 1.60           1.59   
                                        4                 2.09           2.13   
...                                                        ...            ...   
WITCH 5.0   ENGAGE-INDCi2030-1000f-NDCp 595               1.72           1.63   
                                        596               3.10           3.21   
                                        597               2.36           2.43   
                                        598               1.68           1.61   
                                        599               1.92           1.90   

                                                GSAT|CO2_NZCO2|NZCO2  \
model       scenario                    run_id                         
IMAGE 3.0.1 EMF30-ClimatePolicy+SLCF    0                       1.74   
                                        1                       3.29   
                                        2                       1.84   
                                        3                       1.35   
                                        4                       1.71   
...                                                              ...   
WITCH 5.0   ENGAGE-INDCi2030-1000f-NDCp 595                     1.51   
                                        596                     2.79   
                                        597                     1.95   
                                        598                     1.52   
                                        599                     1.71   

                                                GSAT|CO2_NZCO2|2100  \
model       scenario                    run_id                        
IMAGE 3.0.1 EMF30-ClimatePolicy+SLCF    0                      1.73   
                                        1                      3.38   
                                        2                      1.83   
                                        3                      1.33   
                                        4                      1.69   
...                                                             ...   
WITCH 5.0   ENGAGE-INDCi2030-1000f-NDCp 595                    1.44   
                                        596                    2.85   
                                        597                    1.98   
                                        598                    1.50   
                                        599                    1.71   

                                                GSAT|CO2_Negative|NZCO2  \
model       scenario                    run_id                            
IMAGE 3.0.1 EMF30-ClimatePolicy+SLCF    0                           0.0   
                                        1                           0.0   
                                        2                           0.0   
                                        3                           0.0   
                                        4                           0.0   
...                                                                 ...   
WITCH 5.0   ENGAGE-INDCi2030-1000f-NDCp 595                         0.0   
                                        596                         0.0   
                                        597                         0.0   
                                        598                         0.0   
                                        599                         0.0   

                                                GSAT|CO2_Negative|2100  \
model       scenario                    run_id                           
IMAGE 3.0

In [9]:
# Calculate delta (2100 - NZCO2) for each component
components = ["ALL", "CO2_NZCO2", "CO2_Negative", "NonCO2_GHG", "Residual"]

for comp in components:
    metrics_df[f"GSAT|{comp}|delta"] = (
        metrics_df[f"GSAT|{comp}|2100"] - metrics_df[f"GSAT|{comp}|NZCO2"]
    )

# Display delta columns
delta_cols = [col for col in metrics_df.columns if "|delta" in col]
print("Delta (2100 - NZCO2) for each component:")
metrics_df[delta_cols].round(3)

Delta (2100 - NZCO2) for each component:


GSAT|ALL|delta  \
model       scenario                    run_id                   
IMAGE 3.0.1 EMF30-ClimatePolicy+SLCF    0                0.025   
                                        1                0.155   
                                        2                0.028   
                                        3               -0.015   
                                        4                0.034   
...                                                        ...   
WITCH 5.0   ENGAGE-INDCi2030-1000f-NDCp 595             -0.092   
                                        596              0.112   
                                        597              0.072   
                                        598             -0.070   
                                        599             -0.015   

                                                GSAT|CO2_NZCO2|delta  \
model       scenario                    run_id                         
IMAGE 3.0.1 EMF30-ClimatePolicy+SLCF    0                     -0.015   
                                        1                      0.090   
                                        2                     -0.013   
                                        3                     -0.027   
                                        4                     -0.016   
...                                                              ...   
WITCH 5.0   ENGAGE-INDCi2030-1000f-NDCp 595                   -0.076   
                                        596                    0.065   
                                        597                    0.029   
                                        598                   -0.025   
                                        599                    0.005   

                                                GSAT|CO2_Negative|delta  \
model       scenario                    run_id                            
IMAGE 3.0.1 EMF30-ClimatePolicy+SLCF    0                        -0.022   
                                        1                        -0.026   
                                        2                        -0.022   
                                        3                        -0.019   
                                        4                        -0.021   
...                                                                 ...   
WITCH 5.0   ENGAGE-INDCi2030-1000f-NDCp 595                      -0.050   
                                        596                      -0.061   
                                        597                      -0.039   
                                        598                      -0.044   
                                        599                      -0.043   

                                                GSAT|NonCO2_GHG|delta  \
model       scenario                    run_id                          
IMAGE 3.0.1 EMF30-ClimatePolicy+SLCF    0                      -0.005   
                                        1                       0.033   
                                        2                      -0.002   
                                        3                      -0.018   
                                        4                      -0.009   
...                                                               ...   
WITCH 5.0   ENGAGE-INDCi2030-1000f-NDCp 595                    -0.020   
                                        596                    -0.003   
                                        597                    -0.030   
                                        598                    -0.021   
                                        599                    -0.027   

                                                GSAT|Residual|delta  
model       scenario                    run_id                       
IMAGE 3.0.1 EMF30-ClimatePolicy+SLCF    0                     0.067  
                                        1                     0.059  
                                        2     

In [10]:
import matplotlib.pyplot as plt

# Drop validation columns for plotting (only those that exist)
validation_cols = ["sum_components|NZCO2", "diff|NZCO2", "sum_components|2100", "diff|2100", "prop_sum"]
cols_to_drop = [c for c in validation_cols if c in metrics_df.columns]
metrics_df_clean = metrics_df.drop(columns=cols_to_drop)

# Calculate statistics by scenario (across run_id)
stats = metrics_df_clean.groupby(["model", "scenario"]).agg(
    ["median", lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)]
)

# Rename the lambda functions to clearer names
stats.columns = stats.columns.set_levels(["median", "p25", "p75"], level=1)

# Create scenario labels for plotting
scenario_labels = [f"{m}\n{s[:30]}" for m, s in stats.index]

stats.round(2)

GSAT|ALL|NZCO2  \
                                                                                 median   
model                 scenario                                                            
IMAGE 3.0.1           EMF30-ClimatePolicy+SLCF                                     1.83   
MESSAGEix-GLOBIOM 1.1 ENGAGE-NPi2020-1000                                          1.80   
REMIND 3.0            ENGAGE-Feasibility-Maximum-Effort/Enablers & In...           1.59   
REMIND-MAgPIE 2.0-4.1 Diff-1300Gt-hybrid-def                                       1.84   
WITCH 5.0             ENGAGE-INDCi2030-1000f-NDCp                                  1.86   

                                                                                \
                                                                           p25   
model                 scenario                                                   
IMAGE 3.0.1           EMF30-ClimatePolicy+SLCF                            1.58   
MESSAGEix-GLOBIOM 1.1 ENGAGE-NPi2020-1000                                 1.55   
REMIND 3.0            ENGAGE-Feasibility-Maximum-Effort/Enablers & In...  1.42   
REMIND-MAgPIE 2.0-4.1 Diff-1300Gt-hybrid-def                              1.62   
WITCH 5.0             ENGAGE-INDCi2030-1000f-NDCp                         1.58   

                                                                                \
                                                                           p75   
model                 scenario                                                   
IMAGE 3.0.1           EMF30-ClimatePolicy+SLCF                            2.09   
MESSAGEix-GLOBIOM 1.1 ENGAGE-NPi2020-1000                                 2.08   
REMIND 3.0            ENGAGE-Feasibility-Maximum-Effort/Enablers & In...  1.78   
REMIND-MAgPIE 2.0-4.1 Diff-1300Gt-hybrid-def                              2.08   
WITCH 5.0             ENGAGE-INDCi2030-1000f-NDCp                         2.18   

                                                                         GSAT|ALL|2100  \
                                                                                median   
model                 scenario                                                           
IMAGE 3.0.1           EMF30-ClimatePolicy+SLCF                                    1.81   
MESSAGEix-GLOBIOM 1.1 ENGAGE-NPi2020-1000                                         1.77   
REMIND 3.0            ENGAGE-Feasibility-Maximum-Effort/Enablers & In...          1.45   
REMIND-MAgPIE 2.0-4.1 Diff-1300Gt-hybrid-def                                      1.80   
WITCH 5.0             ENGAGE-INDCi2030-1000f-NDCp                                 1.80   

                                                                                \
                                                                           p25   
model                 scenario                                                   
IMAGE 3.0.1           EMF30-ClimatePolicy+SLCF                            1.54   
MESSAGEix-GLOBIOM 1.1 ENGAGE-NPi2020-1000                                 1.50   
REMIND 3.0            ENGAGE-Feasibility-Maximum-Effort/Enablers & In...  1.24   
REMIND-MAgPIE 2.0-4.1 Diff-1300Gt-hybrid-def                              1.55   
WITCH 5.0             ENGAGE-INDCi2030-1000f-NDCp                         1.50   

                                                                                \
                                                                           p75   
model                 scenario                                                   
IMAGE 3.0.1           EMF30-ClimatePolicy+SLCF                            2.11   
MESSAGEix-GLOBIOM 1.1 ENGAGE-NPi2020-1000                                 2.07   
REMIND 3.0            ENGAGE-Feasibility-Maximum-Effort/Enablers & In...  1.69   
REMIND-MAgPIE 2.0-4.1 Diff-1300Gt-hybrid-def                              2.07   
WITCH 5.0             ENGAGE-INDCi2030-1000f-NDCp                         2.17